# Milestone 2: object detection with YOLO
**IE7615 Deep Learning, Group 7.** Builds multi-object images from the single-object photos, trains YOLO26-s and YOLOv8-s, evaluates them, and outputs the **Object ID and bounding box of every object** in an image.

**Input:** `milestone2_outputs/boxes_master.csv` from `M2_Boxes.ipynb`. Test boxes were all checked by a person; training and validation boxes were accepted automatically when independent detectors agreed (measured on the test split: 98.9% correct).

| Step | What it does |
|---|---|
| `compose_quick`, `check_quick`, `train_quick` | Small dry run (50 training grids, 3 epochs) to prove everything works |
| `compose` | Grids of photos side by side at native 224 px, an equal mix of **2x2 (448x448), 3x3 (672x672), 4x4 (896x896), 5x4 (1120x896), 5x5 (1120x1120)**, plus 15% single photos; 1,500 training images; every validation/test photo used exactly once; a second validation/test set of single photos + the five grids; YOLO labels; left-out report per Object ID |
| `check` | No photo crosses splits, every validation/test photo used once, labels valid, preview grids with ground-truth boxes |
| `train_yolo26s`, `train_yolov8s` | Fine-tune COCO-pretrained models at imgsz 1120 (the largest grid), batch 4 for an 8 GB GPU (resumable) |
| `evaluate` | Choose the model on **validation**, then test once: mAP@0.5, mAP@0.5:0.95, precision, recall, per-class AP, results per grid size, fully-correct images, low-quality test, example detections |
| `finetune_yolo26s`, `finetune_yolov8s` | After label corrections: continue from the trained weights for 15 epochs (much faster than retraining); `evaluate` then compares every run |
| `finetune2_yolo26s`, `finetune2_yolov8s` | Second fine-tune from the first one, after single photos were added to training |
| `detect` | Object ID + box table for any image(s): set `M2_DETECT_SOURCE` |

Headless (PowerShell): `$env:M2_RUN="compose_quick,check_quick"; jupyter nbconvert --to notebook --execute M2_Detect.ipynb --output M2_Detect_run.ipynb --ExecutePreprocessor.timeout=-1`

**Safety:** a log per step in `milestone2_outputs/logs/`, the dataset appears only when completely built, run fingerprints (changed inputs are rebuilt, never mixed), resumable training, test data used only after the model is chosen.

## Configuration

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = r"C:\Users\csk23\Documents\Zanthosh\Northeastern\SEMESTERs\4th Sem\Deep Learning\PROJECT - 1"

RUN = {s.strip() for s in os.environ.get("M2_RUN", "").split(",") if s.strip()}
DETECT_SOURCE = os.environ.get("M2_DETECT_SOURCE", "")      # an image file or a folder of images
DETECT_CONF   = os.environ.get("M2_DETECT_CONF", "")        # optional confidence cut-off, e.g. 0.4 for real (non-grid) photos
DETECT_IMGSZ  = os.environ.get("M2_DETECT_IMGSZ", "")       # optional working size, e.g. 448 for single 224-px photos (default 1120)

VALID = {"compose_quick", "check_quick", "train_quick", "compose", "check", "train_yolo26s", "train_yolov8s",
         "finetune_yolo26s", "finetune_yolov8s", "finetune2_yolo26s", "finetune2_yolov8s", "evaluate", "detect"}
assert RUN <= VALID, f"Unknown step(s) in M2_RUN: {RUN - VALID}"
print("Project:", PROJECT_DIR, "| steps to run:", sorted(RUN) or "none (definitions only)")

## Shared settings and helpers (same as M2_Boxes)

In [ ]:
def _common():
    import hashlib, json, os, sys
    from datetime import datetime
    from pathlib import Path

    import numpy as np
    import pandas as pd
    from PIL import Image, ImageOps

    # ----------------------------------------------------------------------------------------------------------
    # Paths
    # ----------------------------------------------------------------------------------------------------------
    PROJECT    = Path(PROJECT_DIR)                # set in the Configuration cell
    DATA_ROOT  = PROJECT / "Dataset - Updated"
    M1_DIR     = PROJECT / "milestone1_outputs_v2"
    CLEAN_LIST = M1_DIR / "clean_dataset.csv"        # 6,942 cleaned photos (Milestone 1, V2 run)
    SPLIT_LIST = M1_DIR / "split.csv"                # Milestone 1 split: train / val / test
    OBJECT_LIST = PROJECT / "object_list.csv"        # course Object List (Google Sheets -> CSV)
    OUT_DIR    = PROJECT / "milestone2_outputs"
    TEXT_BOXES = OUT_DIR / "boxes_auto.csv"          # stage 1  (text prompt)
    VIS_BOXES  = OUT_DIR / "boxes_visual.csv"        # stage 1b (visual prompt)
    REVIEW_DIR = OUT_DIR / "review"                  # stage 4  (one CSV per reviewer)

    IMG_SIZE = 224                                    # every box in this pipeline is in the photo's 224x224 frame

    # Photos that are NOT photos of their object (never boxed, never used in Milestone 2):
    EXCLUDE = {
        "OBJ054_017.jpg",                                         # phone screenshot with personal details
        "OBJ004_007.jpg", "OBJ004_014.jpg", "OBJ004_023.jpg",     # OBJ004 background shots without the toy car
        "OBJ004_030.jpg", "OBJ004_073.jpg",                       # (identified by eye; OBJ004 never tagged them "_bg_")
    }


    # ----------------------------------------------------------------------------------------------------------
    # Logging: every print goes to the console AND a log file
    # ----------------------------------------------------------------------------------------------------------
    class Tee:
        def __init__(self, stream, fh): self.stream, self.fh = stream, fh
        def write(self, s):
            self.stream.write(s)
            try: self.fh.write(s); self.fh.flush()
            except Exception: pass
        def flush(self): self.stream.flush()
        def __getattr__(self, name): return getattr(self.stream, name)

    def _base_stream(s):
        """The real output stream under any logging wrappers (so logs never nest and output stays in the notebook)."""
        while hasattr(s, "fh") and hasattr(s, "stream"): s = s.stream
        return s

    def start_logging(stage, tag=""):
        log_dir = OUT_DIR / "logs"; log_dir.mkdir(parents=True, exist_ok=True)
        path = log_dir / f"{stage}{tag}_{datetime.now():%Y%m%d_%H%M%S}.log"
        fh = open(path, "a", encoding="utf-8")
        sys.stdout, sys.stderr = Tee(_base_stream(sys.stdout), fh), Tee(_base_stream(sys.stderr), fh)
        return path

    def stop_logging():
        for s in (sys.stdout, sys.stderr):
            if hasattr(s, "fh"):
                try: s.fh.close()
                except Exception: pass
        sys.stdout, sys.stderr = _base_stream(sys.stdout), _base_stream(sys.stderr)

    def now(): return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    def step(msg): print(f"\n[{now()}] ===== {msg} =====", flush=True)


    # ----------------------------------------------------------------------------------------------------------
    # Safe file writing and fingerprints
    # ----------------------------------------------------------------------------------------------------------
    def atomic_write_text(path, text):
        path = Path(path); tmp = path.with_name(path.name + ".tmp")
        tmp.write_text(text, encoding="utf-8"); os.replace(tmp, path)

    def atomic_to_csv(df, path):
        path = Path(path); tmp = path.with_name(path.name + ".tmp")
        df.to_csv(tmp, index=False); os.replace(tmp, path)

    def atomic_save_image(img, path, **kw):
        path = Path(path); tmp = path.with_name(path.stem + ".tmp" + path.suffix)
        img.save(tmp, **kw); os.replace(tmp, path)

    def file_md5(path):
        h = hashlib.md5()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
        return h.hexdigest()

    def fingerprint(obj):
        return hashlib.md5(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:12]


    # ----------------------------------------------------------------------------------------------------------
    # Images and boxes
    # ----------------------------------------------------------------------------------------------------------
    def load_224(path):
        """Same rule as Milestone 1 and stages 1/1b: fix rotation (EXIF), RGB, squash (never crop) to 224x224."""
        with Image.open(path) as im:
            im.draft("RGB", (IMG_SIZE * 2, IMG_SIZE * 2))
            im = ImageOps.exif_transpose(im).convert("RGB")
            return im.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BICUBIC)

    def iou(a, b):
        """IoU of two boxes (x1, y1, x2, y2); NaN if either box is missing."""
        if a is None or b is None or any(pd.isna(v) for v in list(a) + list(b)): return np.nan
        iw, ih = min(a[2], b[2]) - max(a[0], b[0]), min(a[3], b[3]) - max(a[1], b[1])
        if iw <= 0 or ih <= 0: return 0.0
        inter = iw * ih
        union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
        return float(inter / union) if union > 0 else 0.0

    def area_frac(b):
        return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1]) / (IMG_SIZE * IMG_SIZE)

    def plausible(b, min_area=0.01, max_area=0.97):
        return b is not None and not any(pd.isna(v) for v in b) and min_area <= area_frac(b) <= max_area

    def photo_table():
        """All photos that need a box: cleaned M1 photos minus EXCLUDE, with split and path."""
        clean = pd.read_csv(CLEAN_LIST)
        split = pd.read_csv(SPLIT_LIST)[["fname", "split"]]
        df = clean.rename(columns={"class_name": "object_id"}).merge(split, on="fname", how="left")
        df = df[~df.fname.isin(EXCLUDE)].reset_index(drop=True)
        df["path"] = [str(DATA_ROOT / f"images_{c}" / f) for c, f in zip(df.object_id, df.fname)]
        if df.split.isna().any():
            raise ValueError(f"{int(df.split.isna().sum())} photos have no split in {SPLIT_LIST}")
        return df[["fname", "object_id", "split", "path"]]

    import types as _t
    return _t.SimpleNamespace(**{k: v for k, v in locals().items() if not k.startswith('__')})

C = _common()
print('C: shared settings and helpers ready')

## Detection pipeline
```
Milestone 2: multi-object images, YOLO training, evaluation and the Object-ID + box output.

INPUT  milestone2_outputs/boxes_master.csv (one box per single-object photo; test boxes all checked by a person)
STEPS  compose  grids of photos side by side (2x2, 3x3, 4x4, 5x4, 5x5 at native 224 px) + YOLO labels
                (no photo crosses splits; every validation/test photo used exactly once); left-out report per Object ID
       check    leakage check, label sanity, preview grids with the ground-truth boxes drawn
       train    fine-tune COCO-pretrained YOLO26-s and YOLOv8-s at imgsz 1120 (resumable)
       evaluate choose the model on VALIDATION, then test once: mAP, precision/recall, per-class AP, task metrics
       detect   Object ID + bounding box for every object in any image you give it
```

In [ ]:
def _detect():
    import csv, json, math, os, re, shutil, sys, time, traceback
    from pathlib import Path

    import numpy as np
    import pandas as pd
    from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageOps


    SCRIPT_VERSION = "m2-detect-v3-shapes"
    TILE = 224                                   # every photo is placed at its native 224x224 size
    IMGSZ = 1120                                 # = the largest grid (5x5), as the course guidance recommends
    DET_DIR    = C.OUT_DIR / "detect"
    DATASET    = DET_DIR / "dataset"
    RUNS       = DET_DIR / "runs"
    FIGS       = DET_DIR / "figures"
    MASTER     = C.OUT_DIR / "boxes_master.csv"
    SEED       = 42

    # Grid design (course guidance: photos side by side, a mix of sizes up to 5x5)
    GRIDS   = [(2, 2), (3, 3), (4, 4), (5, 4), (5, 5)]   # (columns, rows): 448x448 ... 1120x1120 pixels (course sizes)
    EXTRA_SHAPES = [(1, 1)]   # single photos (one object per image), in case single dataset images are given
    EXTRA_SHARE  = 0.15       # share of training images that are single photos (the rest use the five course sizes)
    ALL_SHAPES   = [(1, 1)] + GRIDS   # evaluation of everything that can be given: single photos + the five course grids
    # Visually similar objects, grouped by their names in the course Object List (NOT chosen from test results).
    # In training, half of the multi-object images contain one such group side by side, so the model learns to tell them apart.
    LOOKALIKE_GROUPS = [
        ["OBJ025", "OBJ046", "OBJ065"],               # watch, iWatch, analog watch
        ["OBJ061", "OBJ067"],                         # card wallet, men's wallet
        ["OBJ048", "OBJ057"],                         # wireless power bank, power bank
        ["OBJ030", "OBJ034", "OBJ035", "OBJ049"],     # tennis racket, table-tennis racket, badminton racket, pickleball bat
        ["OBJ014", "OBJ045", "OBJ043"],               # sunglasses, wayfarer sunglasses, swimming goggles
        ["OBJ050", "OBJ059"],                         # hair brush, comb
        ["OBJ002", "OBJ015", "OBJ063"],               # water bottle, water tumbler, measuring bottle
        ["OBJ064", "OBJ070"],                         # RGB projector, astronaut projector
        ["OBJ021", "OBJ024", "OBJ071"],               # toy bunny, plush doll, sloth plushie
        ["OBJ003", "OBJ012"],                         # notebook, computer keyboard (confused in Milestone 1)
    ]
    LOOKALIKE_P = 0.5
    N_TRAIN = 1500            # training grids, an equal share of each size (photos reused with fresh augmentation)
    DEGRADE_P = 0.20          # share of training tiles given one real-photo problem: blur, low resolution, compression, darkness

    MODELS = {"yolo26s": "yolo26s.pt", "yolov8s": "yolov8s.pt"}
    FINETUNE = dict(epochs=15, patience=6, lr0=0.002, warmup_epochs=0.0, close_mosaic=5)   # short continuation on corrected labels
    EVAL_RUNS = ["yolo26s", "yolov8s", "yolo26s_ft", "yolov8s_ft", "yolo26s_ft2", "yolov8s_ft2"]   # every trained run that exists is compared
    TRAIN_ARGS = dict(imgsz=IMGSZ, epochs=60, patience=12, batch=4, workers=2, seed=SEED, deterministic=True,
                      cos_lr=True, mosaic=0.5, close_mosaic=10, mixup=0.0, degrees=0.0, shear=0.0, perspective=0.0,
                      translate=0.1, scale=0.25, fliplr=0.5, flipud=0.0, hsv_h=0.01, hsv_s=0.4, hsv_v=0.3,
                      plots=True, verbose=True)


    # ==========================================================================================================
    # Inputs
    # ==========================================================================================================
    BG_TAG = re.compile(r"(?i)_bg_|BG\d+")

    def usable_photos():
        """Photos with a usable box: test = checked by a person; train/val = boxes accepted automatically."""
        m = pd.read_csv(MASTER)
        ok = (((m.split == "test") & m.status.isin(["verified", "fixed"])) |
              ((m.split != "test") & m.status.isin(["verified", "fixed", "unreviewed_auto"])))
        m = m[ok & m.x1.notna()].copy()
        m["path"] = [str(C.DATA_ROOT / f"images_{c}" / f) for c, f in zip(m.object_id, m.fname)]
        return m[["fname", "object_id", "split", "path", "x1", "y1", "x2", "y2", "status"]].reset_index(drop=True)

    def background_photos():
        """Background-only photos (no object): filename tag _bg_/BG###, not in the cleaned object list, deduplicated,
        plus the OBJ004 background shots found by eye. Each gets ONE split (70/15/15), fixed by a seed."""
        clean = set(pd.read_csv(C.CLEAN_LIST).fname)
        rows = []
        for d in sorted(C.DATA_ROOT.glob("images_OBJ*")):
            for f in sorted(d.iterdir()):
                if f.suffix.lower() not in (".jpg", ".jpeg", ".png", ".mpo", ".heic", ".webp"): continue
                if f.name in clean or f.name == "OBJ054_017.jpg": continue
                if BG_TAG.search(f.name) or f.name in (C.EXCLUDE - {"OBJ054_017.jpg"}):
                    rows.append(dict(fname=f.name, path=str(f), md5=C.file_md5(f)))
        bg = pd.DataFrame(rows).drop_duplicates("md5").sort_values("md5").reset_index(drop=True)
        perm = np.random.default_rng(SEED).permutation(len(bg)); n = len(bg)
        split = np.empty(n, dtype=object)
        split[perm[: int(0.70 * n)]] = "train"; split[perm[int(0.70 * n): int(0.85 * n)]] = "val"; split[perm[int(0.85 * n):]] = "test"
        bg["split"] = split
        return bg


    # ==========================================================================================================
    # Tile operations (each returns the image and the object box in tile coordinates)
    # ==========================================================================================================
    def load_photo(path, size=TILE):
        with Image.open(path) as im:
            im.draft("RGB", (size * 2, size * 2))
            return ImageOps.exif_transpose(im).convert("RGB").resize((size, size), Image.Resampling.BICUBIC)

    def flip_h(img, box, w):
        x1, y1, x2, y2 = box
        return ImageOps.mirror(img), (w - x2, y1, w - x1, y2)

    def colour_jitter(img, rng):
        img = ImageEnhance.Brightness(img).enhance(rng.uniform(0.8, 1.2))
        img = ImageEnhance.Contrast(img).enhance(rng.uniform(0.8, 1.2))
        return ImageEnhance.Color(img).enhance(rng.uniform(0.8, 1.2))

    def degrade(img, rng):
        """One real-photo problem (Milestone 1's weakest condition was low quality)."""
        k = rng.integers(4); w, h = img.size
        if k == 0: return img.filter(ImageFilter.GaussianBlur(rng.uniform(1.0, 2.2)))
        if k == 1:
            f = rng.uniform(0.3, 0.55)
            return img.resize((max(8, int(w * f)), max(8, int(h * f))), Image.Resampling.BILINEAR).resize((w, h), Image.Resampling.BILINEAR)
        if k == 2:
            import io
            b = io.BytesIO(); img.save(b, "JPEG", quality=int(rng.integers(12, 30))); b.seek(0); return Image.open(b).convert("RGB")
        return ImageEnhance.Brightness(img).enhance(rng.uniform(0.45, 0.7))

    def make_grid(objs, cols, rows, rng, augment, fill=()):
        """Photos side by side in a cols x rows grid, each at its native 224 px. Cells beyond the objects (only possible
        at the very end of validation/test) are filled with background photos (no label)."""
        W, H = cols * TILE, rows * TILE
        canvas = Image.new("RGB", (W, H), (128, 128, 128)); labels, tiles = [], []
        items = [("obj", o) for o in objs] + [("bg", f) for f in fill][: cols * rows - len(objs)]
        cells = list(rng.permutation(cols * rows))
        for cell, (kind, it) in zip(cells, items):
            img = load_photo(it["path"]); x0, y0 = (cell % cols) * TILE, (cell // cols) * TILE
            box = (it["x1"], it["y1"], it["x2"], it["y2"]) if kind == "obj" else (0, 0, TILE, TILE)
            if augment:
                if rng.random() < 0.5: img, box = flip_h(img, box, TILE)
                img = colour_jitter(img, rng)
                if rng.random() < DEGRADE_P: img = degrade(img, rng)
            canvas.paste(img, (x0, y0))
            tiles.append(dict(kind=kind, fname=it["fname"], rect=(x0, y0, x0 + TILE, y0 + TILE)))
            if kind == "obj": labels.append(dict(object_id=it["object_id"], fname=it["fname"],
                                                 box=(x0 + box[0], y0 + box[1], x0 + box[2], y0 + box[3])))
        return canvas, labels, tiles

    def yolo_lines(labels, cid, W, H):
        out = []
        for l in labels:
            x1, y1, x2, y2 = l["box"]; x1, x2 = min(W, max(0.0, x1)), min(W, max(0.0, x2)); y1, y2 = min(H, max(0.0, y1)), min(H, max(0.0, y2))
            if x2 - x1 < 4 or y2 - y1 < 4: continue
            out.append(f"{cid[l['object_id']]} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}")
        return out


    # ==========================================================================================================
    # A whole split
    # ==========================================================================================================
    def compose_split(split, photos, bgs, classes, out_root, n_train, rng, shapes=None, out_name=None):
        """train: n_train grids (equal share of each size), photos reused and balanced by class, with augmentation.
        val/test: every usable photo used EXACTLY once, no augmentation; grid sizes cycle so every size is tested."""
        P = photos[photos.split == split].reset_index(drop=True); B = bgs[bgs.split == split].reset_index(drop=True)
        cid = {c: i for i, c in enumerate(classes)}; out = out_name or split; shapes = shapes or GRIDS
        (out_root / "images" / out).mkdir(parents=True, exist_ok=True); (out_root / "labels" / out).mkdir(parents=True, exist_ok=True)
        rows = []; by_cls = {c: list(g.index) for c, g in P.groupby("object_id")}; queues = {c: [] for c in by_cls}
        groups = [[c for c in g if c in by_cls] for g in LOOKALIKE_GROUPS]; groups = [g for g in groups if len(g) >= 2]
        def draw_train(n):                                  # distinct Object IDs per grid, balanced use of every photo
            cls = []
            if n >= 2 and groups and rng.random() < LOOKALIKE_P:          # a look-alike group side by side
                g = groups[int(rng.integers(len(groups)))]; cls = list(rng.permutation(g))[:n]
            rest = [c for c in sorted(by_cls) if c not in cls]
            cls += list(rng.choice(rest, size=min(n - len(cls), len(rest)), replace=False))
            while len(cls) < n: cls.append(rng.choice(sorted(by_cls)))
            out = []
            for c in cls:
                if not queues[c]: queues[c] = list(rng.permutation(by_cls[c]))
                out.append(P.loc[queues[c].pop()])
            return out
        pool = list(rng.permutation(len(P)))
        def draw_once(n):                                   # distinct Object IDs where possible; every photo once
            picked, seen, rest = [], set(), []
            while pool and len(picked) < n:
                i = pool.pop()
                if P.loc[i, "object_id"] in seen: rest.append(i); continue
                picked.append(P.loc[i]); seen.add(P.loc[i, "object_id"])
            pool.extend(rest)
            while pool and len(picked) < n: picked.append(P.loc[pool.pop()])   # end of split: repeats allowed
            return picked
        k = 0
        def save(img, labels, tiles, grid):
            nonlocal k
            name = f"{out}_{k:05d}"; k += 1; W, H = img.size
            img.save(out_root / "images" / out / f"{name}.jpg", quality=95)
            (out_root / "labels" / out / f"{name}.txt").write_text("\n".join(yolo_lines(labels, cid, W, H)) + ("\n" if labels else ""))
            for t in tiles:
                lab = next((l for l in labels if l["fname"] == t["fname"]), None)
                rows.append(dict(image=name, split=out, base_split=split, grid=grid, width=W, height=H, kind=t["kind"], fname=t["fname"],
                                 object_id=lab["object_id"] if lab else "",
                                 **({f"b{j}": round(v, 1) for j, v in zip("1234", lab["box"])} if lab else {})))
        if split == "train":
            n_extra = int(round(EXTRA_SHARE * n_train))
            order = [GRIDS[i % len(GRIDS)] for i in range(n_train - n_extra)] + [EXTRA_SHAPES[i % len(EXTRA_SHAPES)] for i in range(n_extra)]
            order = [order[i] for i in rng.permutation(len(order))]
            for cols, rows_n in order:
                img, labels, tiles = make_grid(draw_train(cols * rows_n), cols, rows_n, rng, augment=True)
                save(img, labels, tiles, f"{cols}x{rows_n}")
        else:
            g = 0; by_size = sorted(shapes, key=lambda gr: gr[0] * gr[1])
            while pool:
                cols, rows_n = shapes[g % len(shapes)]; g += 1
                n = cols * rows_n
                if len(pool) < n:                           # last images: largest shape that the remaining photos fill
                    fit = [gr for gr in by_size if gr[0] * gr[1] <= len(pool)]
                    cols, rows_n = fit[-1] if fit else by_size[0]; n = cols * rows_n
                objs = draw_once(n)
                fill = [B.loc[int(i)] for i in rng.integers(0, len(B), n - len(objs))] if (len(objs) < n and len(B)) else []
                img, labels, tiles = make_grid(objs, cols, rows_n, rng, augment=False, fill=fill)
                save(img, labels, tiles, f"{cols}x{rows_n}")
        return rows


    def left_out_report(photos, out_csv):
        """Per Object ID: photos in the dataset, used, and left out with the reason (course reporting requirement)."""
        clean = pd.read_csv(C.CLEAN_LIST).rename(columns={"class_name": "object_id"})
        master = pd.read_csv(MASTER)
        used = set(photos.fname)
        rows = []
        for oid, g in clean.groupby("object_id"):
            names = set(g.fname); m = master[master.object_id == oid]
            not_obj = len(names & C.EXCLUDE)
            unresolved = int(m[~m.fname.isin(used) & m.status.isin(["unreviewed_none", "pending_fix"])].shape[0])
            excluded = int(m[m.status == "excluded"].shape[0])
            n_used = len(names & used); total = len(names)
            rows.append(dict(object_id=oid, photos=total, used=n_used, left_out=total - n_used, left_out_pct=round(100 * (total - n_used) / max(1, total), 1),
                             not_a_photo_of_the_object=not_obj, excluded_by_reviewer=excluded,
                             no_reliable_box=unresolved))
        df = pd.DataFrame(rows); C.atomic_to_csv(df, out_csv)
        tot = df[["photos", "used", "left_out"]].sum()
        print(f"Left out: {int(tot.left_out)} of {int(tot.photos)} photos ({100 * tot.left_out / tot.photos:.1f}%; course limit 20%) | "
              f"classes above 20%: {df[df.left_out_pct > 20].object_id.tolist()}")
        return df


    def compose_main(quick=False):
        log = C.start_logging("detect_compose", "_quick" if quick else "")
        try:
            C.step(f"Compose multi-object images ({SCRIPT_VERSION})"); print(f"Log file: {log}")
            photos, bgs = usable_photos(), background_photos()
            classes = sorted(pd.read_csv(MASTER).object_id.unique())
            print(f"Usable photos: {photos.split.value_counts().to_dict()} | background photos: {bgs.split.value_counts().to_dict()}")
            DET_DIR.mkdir(parents=True, exist_ok=True)
            left_out_report(photos, DET_DIR / "left_out_per_object.csv")
            sig = C.fingerprint(dict(v=SCRIPT_VERSION, master=C.file_md5(MASTER), n=N_TRAIN, grids=GRIDS, extra=EXTRA_SHAPES,
                                     share=EXTRA_SHARE, all=ALL_SHAPES, deg=DEGRADE_P, look=LOOKALIKE_GROUPS, lp=LOOKALIKE_P,
                                     quick=quick, bgs=sorted(bgs.md5)))
            root = DATASET.with_name(DATASET.name + ("_quick" if quick else ""))
            marker = root / "DATASET_OK.json"
            if marker.exists() and json.loads(marker.read_text(encoding="utf-8")).get("sig") == sig:
                print(f"Dataset already built for fingerprint {sig}: {root}"); return root
            tmp = root.with_name(root.name + "_building")
            if tmp.exists(): shutil.rmtree(tmp)
            rng = np.random.default_rng(SEED); rows = []
            for split in ("train", "val", "test"):
                t0 = time.time()
                r = compose_split(split, photos, bgs, classes, tmp, 50 if quick else N_TRAIN, rng)
                rows += r
                imgs = len({x["image"] for x in r}); nobj = sum(1 for x in r if x["kind"] == "obj")
                print(f"  {split}: {imgs} images, {nobj} object boxes ({time.time() - t0:.0f}s)")
            for split in ("val", "test"):                     # singles+grids evaluation sets (same photos, own images)
                r = compose_split(split, photos, bgs, classes, tmp, 0, rng, shapes=ALL_SHAPES, out_name=f"{split}_shapes")
                rows += r; print(f"  {split}_shapes: {len({x['image'] for x in r})} images (single photos + the five course grids)")
            man = pd.DataFrame(rows)
            (tmp / "data_shapes.yaml").write_text(f"path: {root.as_posix()}\ntrain: images/train\nval: images/val_shapes\ntest: images/test_shapes\nnames:\n"
                                                  + "".join(f"  {i}: {c}\n" for i, c in enumerate(classes)), encoding="utf-8")
            (tmp / "data.yaml").write_text(f"path: {root.as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n"
                                           + "".join(f"  {i}: {c}\n" for i, c in enumerate(classes)), encoding="utf-8")
            man.to_csv(tmp / "manifest.csv", index=False)
            C.atomic_write_text(tmp / "DATASET_OK.json", json.dumps(dict(sig=sig, built=C.now(), counts=man.groupby("split").image.nunique().to_dict(),
                                                                         grids=man.drop_duplicates("image").groupby(["split", "grid"]).size().unstack(fill_value=0).to_dict("index")), indent=2))
            if root.exists(): shutil.rmtree(root)
            os.replace(tmp, root)                                # the dataset appears only when it is complete
            print(f"Dataset ready: {root}")
            return root
        finally:
            C.stop_logging()


    # ==========================================================================================================
    # Check: leakage, label sanity, previews
    # ==========================================================================================================
    def check_main(quick=False):
        root = DATASET.with_name(DATASET.name + ("_quick" if quick else ""))
        log = C.start_logging("detect_check", "_quick" if quick else "")
        try:
            C.step("Check the composite dataset"); print(f"Log file: {log}")
            man = pd.read_csv(root / "manifest.csv", dtype={"object_id": str}).fillna({"object_id": "", "fname": ""})
            names = [l.split(": ", 1)[1] for l in (root / "data.yaml").read_text(encoding="utf-8").splitlines() if re.match(r"\s+\d+: ", l)]
            problems = []
            if "base_split" not in man.columns: man["base_split"] = man.split
            used = man[man.kind.isin(["obj", "bg"])].groupby("fname").base_split.nunique()
            cross = used[used > 1]
            if len(cross): problems.append(f"{len(cross)} source photos appear in more than one split, e.g. {list(cross.index[:3])}")
            evals = [x for x in ("val", "test", "val_shapes", "test_shapes") if x in set(man.split)]
            for sp in evals:
                o = man[(man.split == sp) & (man.kind == "obj")]
                if o.fname.duplicated().any(): problems.append(f"{sp}: {int(o.fname.duplicated().sum())} photos used more than once")
            master = usable_photos()
            for sp in evals:
                base = sp.replace("_shapes", "")
                need = set(master[master.split == base].fname); got = set(man[(man.split == sp) & (man.kind == "obj")].fname)
                print(f"{sp}: {len(got)} of {len(need)} usable photos placed ({100 * len(got) / max(1, len(need)):.1f}%)")
                if len(got) < len(need): problems.append(f"{sp}: only {len(got)} of {len(need)} photos placed")
            bad_lbl = 0; n_lbl = 0
            for sp in ["train"] + evals:
                for f in (root / "labels" / sp).glob("*.txt"):
                    for line in f.read_text().split("\n"):
                        if not line.strip(): continue
                        n_lbl += 1; p = line.split()
                        ok = len(p) == 5 and 0 <= int(p[0]) < len(names) and all(0 <= float(v) <= 1 for v in p[1:]) and float(p[3]) > 0 and float(p[4]) > 0
                        bad_lbl += not ok
            if bad_lbl: problems.append(f"{bad_lbl} malformed label lines")
            stats = man[man.kind == "obj"].groupby("split").object_id.value_counts().groupby(level=0).agg(["min", "max"])
            print(f"Label lines: {n_lbl} | malformed: {bad_lbl}\nObjects per class (min/max):\n{stats}")
            print(f"Images per grid size:\n{man.drop_duplicates('image').groupby(['split', 'grid']).size().unstack(fill_value=0)}")
            FIGS.mkdir(parents=True, exist_ok=True)
            for sp in ["train"] + evals:
                sheet = Image.new("RGB", (3 * 448, 2 * 448), "white")
                first = man[man.split == sp].drop_duplicates("grid").image.tolist()     # one example of every grid size
                ims = [root / "images" / sp / f"{n}.jpg" for n in first][:6]
                for i, p in enumerate(ims):
                    im = Image.open(p).convert("RGB"); d = ImageDraw.Draw(im); W, H = im.size
                    lf = root / "labels" / sp / (p.stem + ".txt")
                    for line in lf.read_text().split("\n"):
                        if not line.strip(): continue
                        c, cx, cy, w, h = line.split(); cx, w = float(cx) * W, float(w) * W; cy, h = float(cy) * H, float(h) * H
                        d.rectangle([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], outline=(0, 255, 0), width=3)
                        d.text((cx - w / 2 + 3, cy - h / 2 + 2), names[int(c)], fill=(0, 255, 0))
                    im.thumbnail((448, 448)); sheet.paste(im, ((i % 3) * 448, (i // 3) * 448))
                C.atomic_save_image(sheet, FIGS / f"composites_{sp}{'_quick' if quick else ''}.jpg", quality=90)
            verdict = "PASSED" if not problems else "FAILED: " + "; ".join(problems)
            C.atomic_write_text(root / "CHECK.txt", verdict); print(f"\nDataset check {verdict}\nPreview grids: {FIGS}")
            if problems: raise RuntimeError(verdict)
        finally:
            C.stop_logging()


    # ==========================================================================================================
    # Train (resumable)
    # ==========================================================================================================
    def finetune_main(arch, base_run=None, run_name=None):
        """Continue from runs/<base_run>/weights/best.pt on the CURRENT dataset for a few epochs (label corrections,
        new training shapes). Defaults: base_run = arch, run_name = arch + "_ft"."""
        from ultralytics import YOLO
        log = C.start_logging(f"detect_finetune_{arch}")
        try:
            C.step(f"Fine-tune {arch} on the current labels"); print(f"Log file: {log}")
            root = DATASET; base = RUNS / (base_run or arch) / "weights" / "best.pt"
            if not base.exists(): raise RuntimeError(f"{base} not found: train {arch} first")
            if not (root / "CHECK.txt").read_text().startswith("PASSED"): raise RuntimeError("Run the check step first (it must PASS)")
            sig = C.fingerprint(dict(ds=json.loads((root / "DATASET_OK.json").read_text())["sig"], base=C.file_md5(base), ft=FINETUNE, args=TRAIN_ARGS))
            name = run_name or f"{arch}_ft"; run = RUNS / name; done, last = run / "DONE.json", run / "weights" / "last.pt"
            if done.exists() and json.loads(done.read_text()).get("sig") == sig:
                print(f"{name} already done for fingerprint {sig}"); return run
            if run.exists() and not ((run / "SIG").exists() and (run / "SIG").read_text().strip() == sig):
                stale = RUNS / "_stale" / f"{name}_{C.now().replace(':', '').replace(' ', '_')}"; stale.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(run), str(stale)); print(f"!! old run moved to {stale}")
            t0 = time.time()
            if last.exists():
                print(f"Resuming {name}"); YOLO(str(last)).train(resume=True)
            else:
                run.mkdir(parents=True, exist_ok=True); (run / "SIG").write_text(sig)
                args = dict(TRAIN_ARGS, **FINETUNE); print(f"Fine-tuning from {base} with {json.dumps(args)}")
                YOLO(str(base)).train(data=str(root / "data.yaml"), project=str(RUNS), name=name, exist_ok=True, **args)
            C.atomic_write_text(done, json.dumps(dict(sig=sig, finished=C.now(), minutes=round((time.time() - t0) / 60, 1)), indent=2))
            return run
        finally:
            C.stop_logging()


    def train_main(arch, quick=False):
        from ultralytics import YOLO
        root = DATASET.with_name(DATASET.name + ("_quick" if quick else ""))
        log = C.start_logging(f"detect_train_{arch}", "_quick" if quick else "")
        try:
            C.step(f"Train {arch}"); print(f"Log file: {log}")
            if not (root / "CHECK.txt").exists() or not (root / "CHECK.txt").read_text().startswith("PASSED"):
                raise RuntimeError("Run the check step first (it must PASS)")
            sig = C.fingerprint(dict(ds=json.loads((root / "DATASET_OK.json").read_text())["sig"], arch=arch, args=TRAIN_ARGS, quick=quick))
            name = arch + ("_quick" if quick else ""); run = RUNS / name
            done, last = run / "DONE.json", run / "weights" / "last.pt"
            if done.exists() and json.loads(done.read_text()).get("sig") == sig:
                print(f"{arch} already trained for fingerprint {sig}"); return run
            if run.exists() and not ((run / "SIG").exists() and (run / "SIG").read_text().strip() == sig):
                stale = RUNS / "_stale" / f"{name}_{C.now().replace(':', '').replace(' ', '_')}"; stale.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(run), str(stale)); print(f"!! old run belongs to other data/settings -> moved to {stale}")
            t0 = time.time()
            if last.exists():
                print(f"Resuming {arch} from {last}"); YOLO(str(last)).train(resume=True)
            else:
                run.mkdir(parents=True, exist_ok=True); (run / "SIG").write_text(sig)
                args = dict(TRAIN_ARGS, epochs=3 if quick else TRAIN_ARGS["epochs"])
                print(f"Training {MODELS[arch]} with {json.dumps(args)}")
                YOLO(MODELS[arch]).train(data=str(root / "data.yaml"), project=str(RUNS), name=name, exist_ok=True, **args)
            C.atomic_write_text(done, json.dumps(dict(sig=sig, finished=C.now(), minutes_this_session=round((time.time() - t0) / 60, 1)), indent=2))
            return run
        finally:
            C.stop_logging()


    # ==========================================================================================================
    # Evaluate
    # ==========================================================================================================
    def read_gt(root, split):
        gt = {}
        for f in (root / "labels" / split).glob("*.txt"):
            rows = []; W, H = Image.open(root / "images" / split / (f.stem + ".jpg")).size
            for line in f.read_text().split("\n"):
                if not line.strip(): continue
                c, cx, cy, w, h = line.split(); cx, w = float(cx) * W, float(w) * W; cy, h = float(cy) * H, float(h) * H
                rows.append((int(c), (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)))
            gt[f.stem] = rows
        return gt

    def match_image(gts, preds, iou_thr=0.5):
        """Greedy matching by confidence. preds: list of (cls, conf, box). Returns TP (right ID and place), wrong-ID hits,
        misses and false positives."""
        used, tp, wrong = set(), 0, 0
        for c, conf, b in sorted(preds, key=lambda p: -p[1]):
            best, bi = 0.0, None
            for i, (gc, gb) in enumerate(gts):
                if i in used: continue
                v = C.iou(b, gb)
                if v > best: best, bi = v, i
            if bi is not None and best >= iou_thr:
                used.add(bi)
                if gts[bi][0] == c: tp += 1
                else: wrong += 1
        fp = len(preds) - tp - wrong
        return dict(n_gt=len(gts), n_pred=len(preds), tp=tp, wrong_id=wrong, miss=len(gts) - tp - wrong, fp=fp,
                    fully_correct=int(tp == len(gts) and len(preds) == tp))

    def one_per_cell(preds):
        """Grid images only: every 224x224 cell holds exactly one photo (one object), so keep the most confident box
        whose centre lies in each cell."""
        best = {}
        for p in sorted(preds, key=lambda q: -q[1]):
            cx, cy = (p[2][0] + p[2][2]) / 2, (p[2][1] + p[2][3]) / 2
            cell = (int(cx // TILE), int(cy // TILE))
            if cell not in best: best[cell] = p
        return list(best.values())

    def dedupe(preds, iou_thr):
        """Post-processing. iou_thr = None: raw model output. A number: keep the most confident box and drop any
        other box overlapping it by >= iou_thr (any Object ID). "cell": one box per 224x224 grid cell."""
        if iou_thr is None: return preds
        if iou_thr == "cell": return one_per_cell(preds)
        keep = []
        for p in sorted(preds, key=lambda q: -q[1]):
            if all(C.iou(p[2], k[2]) < iou_thr for k in keep): keep.append(p)
        return keep

    def task_metrics(gt, pred, conf, iou_thr=None):
        per = []
        for img, gts in gt.items():
            ps = dedupe([p for p in pred.get(img, []) if p[1] >= conf], iou_thr)
            per.append(dict(image=img, **match_image(gts, ps)))
        return pd.DataFrame(per)

    def summarise_task(df):
        g = df.n_gt.sum()
        return dict(images=int(len(df)), fully_correct=round(float(df.fully_correct.mean()), 4),
                    object_recall=round(float(df.tp.sum() / max(1, g)), 4),
                    precision=round(float(df.tp.sum() / max(1, df.n_pred.sum())), 4),
                    wrong_id=int(df.wrong_id.sum()), missed=int(df.miss.sum()), false_pos=int(df.fp.sum()))

    def bootstrap_ci(values, n=2000, seed=SEED):
        v = np.asarray(values, float); rng = np.random.default_rng(seed)
        means = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(n)]
        return [round(float(np.percentile(means, 2.5)), 4), round(float(np.percentile(means, 97.5)), 4)]

    def choose_conf(gt_val, pred_val, iou_thr=None):
        grid = np.round(np.arange(0.10, 0.91, 0.05), 2); best = (0.25, -1.0)
        for c in grid:
            fc = task_metrics(gt_val, pred_val, c, iou_thr).fully_correct.mean()
            if fc > best[1]: best = (float(c), float(fc))
        return best

    def choose_postprocessing(gt_val, pred_val):
        """Choose the duplicate-removal overlap and the confidence threshold together, on VALIDATION only."""
        best = None
        for thr in (None, 0.7, 0.6, 0.5, 0.4, 0.3, "cell"):
            c, fc = choose_conf(gt_val, pred_val, thr)
            if best is None or fc > best[2] + 1e-9: best = (thr, c, fc)
        return best

    def to_canvas(img):
        """Natural-size inference: a small image (single photo or small grid) is placed unscaled at the top-left of an
        IMGSZ x IMGSZ grey canvas, so every 224-px photo is seen at the size used in training. Larger images unchanged."""
        W, H = img.size
        if W > IMGSZ or H > IMGSZ: return img, False
        c = Image.new("RGB", (IMGSZ, IMGSZ), (128, 128, 128)); c.paste(img, (0, 0)); return c, True

    def predict_images(model, paths, mode, conf=0.05):
        """Predictions for a list of image paths. mode "standard": YOLO's usual resize; "native": to_canvas first.
        Returns {stem: [(cls, conf, (x1, y1, x2, y2)), ...]} in original-image pixels."""
        out = {}
        for s0 in range(0, len(paths), 16):
            chunk = paths[s0:s0 + 16]
            if mode == "native":
                ims = [Image.open(p).convert("RGB") for p in chunk]; can = [to_canvas(im) for im in ims]
                res = model.predict([c for c, _ in can], imgsz=IMGSZ, conf=conf, max_det=60, verbose=False)
                for p, im, (c, padded), r in zip(chunk, ims, can, res):
                    W, H = im.size; keep = []
                    for cl, pr, b in zip(r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.xyxy.cpu().numpy()):
                        x1, y1, x2, y2 = map(float, b)
                        if padded:
                            if (x1 + x2) / 2 > W or (y1 + y2) / 2 > H: continue      # a box in the grey padding
                            x2, y2 = min(x2, W), min(y2, H)
                        keep.append((int(cl), float(pr), (x1, y1, x2, y2)))
                    out[Path(p).stem] = keep
            else:
                for r in model.predict(chunk, imgsz=IMGSZ, conf=conf, max_det=60, verbose=False):
                    out[Path(r.path).stem] = [(int(c), float(q), tuple(map(float, b))) for c, q, b in
                                              zip(r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.xyxy.cpu().numpy())]
        return out

    def predict_split(model_path, root, split):
        from ultralytics import YOLO
        model = YOLO(str(model_path)); out = {}
        paths = sorted(str(p) for p in (root / "images" / split).glob("*.jpg"))
        for s in range(0, len(paths), 32):
            for r in model.predict(paths[s:s + 32], imgsz=IMGSZ, conf=0.05, max_det=50, verbose=False):
                out[Path(r.path).stem] = [(int(c), float(p), tuple(map(float, b))) for c, p, b in
                                          zip(r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.xyxy.cpu().numpy())]
        return out

    def make_low_quality(root):
        """A degraded copy of the test images (1/4 resolution, darker, heavy JPEG) with the same labels."""
        lq = DET_DIR / "dataset_lowq"
        if (lq / "OK").exists(): return lq
        if lq.exists(): shutil.rmtree(lq)
        (lq / "images" / "test").mkdir(parents=True); shutil.copytree(root / "labels" / "test", lq / "labels" / "test")
        for p in (root / "images" / "test").glob("*.jpg"):
            im = Image.open(p).convert("RGB"); w, h = im.size
            im = im.resize((w // 4, h // 4), Image.Resampling.BILINEAR).resize((w, h), Image.Resampling.BILINEAR)
            ImageEnhance.Brightness(im).enhance(0.6).save(lq / "images" / "test" / p.name, quality=20)
        names = (root / "data.yaml").read_text(encoding="utf-8").split("names:", 1)[1]
        (lq / "data.yaml").write_text(f"path: {lq.as_posix()}\ntrain: images/test\nval: images/test\ntest: images/test\nnames:{names}", encoding="utf-8")
        (lq / "OK").write_text("ok"); return lq

    def evaluate_main():
        from ultralytics import YOLO
        log = C.start_logging("detect_evaluate")
        try:
            C.step("Evaluate: choose on validation, then test once"); print(f"Log file: {log}")
            root = DATASET; FIGS.mkdir(parents=True, exist_ok=True)
            names = [l.split(": ", 1)[1] for l in (root / "data.yaml").read_text(encoding="utf-8").splitlines() if re.match(r"\s+\d+: ", l)]
            man = pd.read_csv(root / "manifest.csv").drop_duplicates("image").set_index("image")
            gt_val, gt_test = read_gt(root, "val"), read_gt(root, "test")
            res = {}
            for arch in EVAL_RUNS:
                w = RUNS / arch / "weights" / "best.pt"
                if not w.exists(): continue
                m = YOLO(str(w))
                v = m.val(data=str(root / "data.yaml"), split="val", imgsz=IMGSZ, batch=8, plots=False, verbose=False,
                          project=str(DET_DIR / "eval"), name=f"{arch}_val", exist_ok=True)
                res[arch] = dict(val_map50=round(float(v.box.map50), 4), val_map=round(float(v.box.map), 4), weights=str(w))
                print(f"{arch}: validation mAP50 {v.box.map50:.4f} | mAP50-95 {v.box.map:.4f}")
            if not res: raise RuntimeError("No trained model found: run the train steps first")
            best = max(res, key=lambda a: res[a]["val_map"])
            print(f"Selected on VALIDATION mAP50-95: {best}")
            for arch in res:                                     # test numbers for the table (selection is already fixed)
                m = YOLO(res[arch]["weights"])
                t = m.val(data=str(root / "data.yaml"), split="test", imgsz=IMGSZ, batch=8, plots=(arch == best), verbose=False,
                          project=str(DET_DIR / "eval"), name=f"{arch}_test", exist_ok=True)
                res[arch].update(test_map50=round(float(t.box.map50), 4), test_map=round(float(t.box.map), 4),
                                 test_precision=round(float(t.box.mp), 4), test_recall=round(float(t.box.mr), 4))
                if arch == best:
                    ap = pd.DataFrame(dict(object_id=[names[i] for i in t.box.ap_class_index], ap50=np.round(t.box.ap50, 4), ap50_95=np.round(t.box.ap, 4)))
                    C.atomic_to_csv(ap.sort_values("ap50_95"), DET_DIR / "per_class_ap_test.csv")
            # task-level metrics with a confidence threshold chosen on validation
            bw = res[best]["weights"]
            pv = predict_split(bw, root, "val")
            conf_raw, _ = choose_conf(gt_val, pv)                         # raw model output (no duplicate removal)
            dup_iou, conf, fc_val = choose_postprocessing(gt_val, pv)     # duplicate removal + threshold, VALIDATION only
            print(f"Post-processing chosen on validation: duplicate-removal IoU {dup_iou}, confidence >= {conf} "
                  f"(validation fully-correct {fc_val:.3f})")
            pt = predict_split(bw, root, "test")
            raw = task_metrics(gt_test, pt, conf_raw); raw["grid"] = raw.image.map(man.grid)
            tm = task_metrics(gt_test, pt, conf, dup_iou); tm["grid"] = tm.image.map(man.grid)
            task = summarise_task(tm); task["fully_correct_ci95"] = bootstrap_ci(tm.fully_correct)
            task["conf_threshold"] = conf; task["duplicate_removal_iou"] = dup_iou
            task["by_grid"] = {k: summarise_task(g) for k, g in tm.groupby("grid")}
            task["raw_model_output"] = dict(conf_threshold=conf_raw, **summarise_task(raw),
                                            fully_correct_ci95=bootstrap_ci(raw.fully_correct),
                                            by_grid={k: summarise_task(g) for k, g in raw.groupby("grid")})
            C.atomic_to_csv(tm, DET_DIR / "task_metrics_per_image_test.csv")
            # robustness: low-quality copy of the test set
            lq = make_low_quality(root)
            r = YOLO(bw).val(data=str(lq / "data.yaml"), split="test", imgsz=IMGSZ, batch=8, plots=False, verbose=False,
                             project=str(DET_DIR / "eval"), name=f"{best}_lowq", exist_ok=True)
            task["low_quality_map50"] = round(float(r.box.map50), 4); task["low_quality_map"] = round(float(r.box.map), 4)
            # example detections figure
            for gname, g in tm.groupby("grid"):                  # one full-resolution example per grid size
                img = g.image.iloc[0]
                im = Image.open(root / "images" / "test" / f"{img}.jpg").convert("RGB"); d = ImageDraw.Draw(im)
                for c, b in gt_test[img]: d.rectangle(b, outline=(0, 200, 0), width=2)
                for c, p, b in dedupe([x for x in pt.get(img, []) if x[1] >= conf], dup_iou):
                    d.rectangle(b, outline=(255, 40, 40), width=3); d.text((b[0] + 3, b[1] + 2), f"{names[c]} {p:.2f}", fill=(255, 255, 0))
                C.atomic_save_image(im, FIGS / f"example_detections_test_{gname}.jpg", quality=92)
            # ---- single photos + the five course grids: method and post-processing chosen on val_shapes, reported on test_shapes
            any_shape = None
            if (root / "images" / "val_shapes").exists():
                m = YOLO(bw); gvs, gts = read_gt(root, "val_shapes"), read_gt(root, "test_shapes")
                vpaths = sorted(str(p) for p in (root / "images" / "val_shapes").glob("*.jpg"))
                tpaths = sorted(str(p) for p in (root / "images" / "test_shapes").glob("*.jpg"))
                man_s = pd.read_csv(root / "manifest.csv").drop_duplicates("image").set_index("image")
                cand = {}
                for mode in ("standard", "native"):
                    pv_s = predict_images(m, vpaths, mode); thr, c, fc = choose_postprocessing(gvs, pv_s)
                    cand[mode] = dict(dup=thr, conf=c, val_fully_correct=round(fc, 4))
                    print(f"Singles+grids validation, {mode} inference: post-processing {thr}, conf {c}, fully correct {fc:.3f}")
                mode = max(cand, key=lambda k: cand[k]["val_fully_correct"])
                per_mode = {}
                for md in ("standard", "native"):
                    pt_s = predict_images(m, tpaths, md); tms = task_metrics(gts, pt_s, cand[md]["conf"], cand[md]["dup"])
                    tms["shape"] = tms.image.map(man_s.grid)
                    per_mode[md] = dict(**summarise_task(tms), fully_correct_ci95=bootstrap_ci(tms.fully_correct),
                                        by_shape={k: summarise_task(g) for k, g in tms.groupby("shape")})
                    if md == mode: C.atomic_to_csv(tms, DET_DIR / "task_metrics_per_image_test_shapes.csv")
                sv = m.val(data=str(root / "data_shapes.yaml"), split="test", imgsz=IMGSZ, batch=8, plots=False, verbose=False,
                           project=str(DET_DIR / "eval"), name=f"{best}_shapes_test", exist_ok=True)
                any_shape = dict(selected_inference=mode, selection_rule="highest singles+grids VALIDATION fully-correct",
                                 candidates=cand, test=per_mode[mode], test_other_method=per_mode[[k for k in per_mode if k != mode][0]],
                                 test_map50_standard=round(float(sv.box.map50), 4), test_map_standard=round(float(sv.box.map), 4))
                print(f"Singles+grids inference chosen on validation: {mode} | test fully-correct {per_mode[mode]['fully_correct']} | "
                      f"recall {per_mode[mode]['object_recall']} | precision {per_mode[mode]['precision']}")
            summary = dict(selected_model=best, selection_rule="highest validation mAP50-95", models=res, test_task_metrics=task,
                           any_shape=any_shape, finished=C.now())
            C.atomic_write_text(DET_DIR / "results.json", json.dumps(summary, indent=2, default=str))
            lines = ["# Milestone 2 results", f"Selected on validation: **{best}**", "",
                     "| Model | Val mAP50 | Val mAP50-95 | Test mAP50 | Test mAP50-95 | Test precision | Test recall |", "|---|---|---|---|---|---|---|"]
            lines += [f"| {a} | {r['val_map50']} | {r['val_map']} | {r['test_map50']} | {r['test_map']} | {r['test_precision']} | {r['test_recall']} |" for a, r in res.items()]
            lines += ["", f"Task metrics on test (confidence >= {conf} and duplicate removal at IoU {dup_iou}, both chosen on "
                          f"validation; the raw model output is under 'raw_model_output'): " + json.dumps(task, default=str),
                      "", "Single photos + the five course grids (inference and post-processing chosen on validation): " + json.dumps(any_shape, default=str),
                      "", "Left out per Object ID: detect/left_out_per_object.csv"]
            C.atomic_write_text(DET_DIR / "results_summary.md", "\n".join(lines)); print("\n".join(lines))
        finally:
            C.stop_logging()


    # ==========================================================================================================
    # Detect: Object ID + box for every object in any image
    # ==========================================================================================================
    def detect_main(source, conf=None, imgsz=None):
        from ultralytics import YOLO
        log = C.start_logging("detect_run")
        try:
            C.step(f"Detect objects in {source}"); print(f"Log file: {log}")
            summ = json.loads((DET_DIR / "results.json").read_text())
            weights = summ["models"][summ["selected_model"]]["weights"]
            a = summ.get("any_shape") or {}
            mode = a.get("selected_inference", "standard")
            cfg = (a.get("candidates") or {}).get(mode) or {}
            conf = conf if conf is not None else cfg.get("conf", summ["test_task_metrics"]["conf_threshold"])
            dup_iou = cfg.get("dup", summ["test_task_metrics"].get("duplicate_removal_iou")) if cfg else summ["test_task_metrics"].get("duplicate_removal_iou")
            if imgsz: mode = "standard"                                      # an explicit size overrides natural-size inference
            imgsz = int(imgsz) if imgsz else IMGSZ
            print(f"Model {summ['selected_model']} | inference {mode} | confidence >= {conf} | post-processing {dup_iou} | image size {imgsz}")
            out_dir = DET_DIR / "detections"; out_dir.mkdir(parents=True, exist_ok=True)
            model = YOLO(weights); rows = []
            src = Path(source)
            paths = sorted(str(p) for p in src.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp")) if src.is_dir() else [str(src)]
            if mode == "native" and imgsz == IMGSZ: preds = predict_images(model, paths, "native", conf=conf)
            else:
                preds = {}
                for r in model.predict(paths, imgsz=imgsz, conf=conf, max_det=60, verbose=False):
                    preds[Path(r.path).stem] = [(int(c), float(q), tuple(map(float, b))) for c, q, b in
                                                zip(r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.xyxy.cpu().numpy())]
            for pth in paths:
                name = Path(pth).name; im = Image.open(pth).convert("RGB"); d = ImageDraw.Draw(im); W0, H0 = im.size
                pm = dup_iou if (dup_iou != "cell" or (W0 % TILE == 0 and H0 % TILE == 0)) else None   # "cell" only for real grids
                dets = dedupe([p for p in preds.get(Path(pth).stem, []) if p[1] >= conf], pm)
                print(f"\nImage: {name}   ({len(dets)} objects found)\n  Object ID   Confidence   Box (x1, y1, x2, y2) in pixels")
                for c, p, b in sorted(dets, key=lambda t: (t[2][1], t[2][0])):
                    oid = model.names[int(c)]; x1, y1, x2, y2 = [int(round(v)) for v in b]
                    print(f"  {oid:<10}  {p:.2f}         ({x1:4d}, {y1:4d}, {x2:4d}, {y2:4d})")
                    rows.append(dict(image=name, object_id=oid, confidence=round(float(p), 3), x1=x1, y1=y1, x2=x2, y2=y2))
                    d.rectangle([x1, y1, x2, y2], outline=(255, 40, 40), width=3); d.text((x1 + 3, y1 + 2), f"{oid} {p:.2f}", fill=(255, 255, 0))
                im.save(out_dir / f"detected_{Path(name).stem}.jpg", quality=92)
            C.atomic_to_csv(pd.DataFrame(rows), out_dir / "detections.csv")
            C.atomic_write_text(out_dir / "detections.json", json.dumps(rows, indent=2))
            print(f"\nSaved: {out_dir / 'detections.csv'}, detections.json and drawn images")
        finally:
            C.stop_logging()

    import types as _t
    return _t.SimpleNamespace(**{k: v for k, v in locals().items() if not k.startswith('__')})

DT = _detect()
print('DT: detection pipeline ready')

# Run
Each cell runs only if its step is in `RUN`.

In [ ]:
def run_step(name, fn, **kw):
    if name not in RUN:
        print(f"[skip] {name}"); return
    print(f"[run ] {name} {kw}")
    try:
        fn(**kw)
    finally:
        C.stop_logging()

In [ ]:
run_step("compose_quick", DT.compose_main, quick=True)
run_step("check_quick", DT.check_main, quick=True)
run_step("train_quick", DT.train_main, arch="yolo26s", quick=True)

In [ ]:
run_step("compose", DT.compose_main)
run_step("check", DT.check_main)

In [ ]:
run_step("train_yolo26s", DT.train_main, arch="yolo26s")
run_step("train_yolov8s", DT.train_main, arch="yolov8s")

In [ ]:
run_step("finetune_yolo26s", DT.finetune_main, arch="yolo26s")
run_step("finetune_yolov8s", DT.finetune_main, arch="yolov8s")

In [ ]:
# second fine-tune: continues from the first fine-tune (adds single photos to training)
run_step("finetune2_yolo26s", DT.finetune_main, arch="yolo26s", base_run="yolo26s_ft", run_name="yolo26s_ft2")
run_step("finetune2_yolov8s", DT.finetune_main, arch="yolov8s", base_run="yolov8s_ft", run_name="yolov8s_ft2")

In [ ]:
run_step("evaluate", DT.evaluate_main)

In [ ]:
run_step("detect", DT.detect_main, source=DETECT_SOURCE or str(DT.DATASET / "images" / "test"),
         conf=float(DETECT_CONF) if DETECT_CONF else None, imgsz=int(DETECT_IMGSZ) if DETECT_IMGSZ else None)